In [307]:
import random as rd
from tqdm import tqdm

import numpy as np

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import Lowercase

from transformers import PreTrainedTokenizerFast

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset

### Dataset preparation

In [18]:
raw_dataset = load_dataset("tanaos/synthetic-NER-dataset-v1")
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 8716
    })
})

In [14]:
raw_train_dataset = raw_dataset["train"]
raw_train_dataset

Dataset({
    features: ['text', 'labels'],
    num_rows: 8716
})

In [47]:
clean_labels = []
for label_string in raw_train_dataset["labels"]:
    clean_labels.append([item.rstrip(']"').lstrip('"[').strip("'") for item in label_string.split(", ")])

In [55]:
max_length = 0
for cls_list in clean_labels:
    max_length = max(max_length, len(cls_list))
max_length

33

In [61]:
splitted_text = []
for i, raw_text in enumerate(raw_train_dataset["text"]):
    splitted = raw_text.split()
    splitted_text.append(splitted)
    assert len(clean_labels[i]) == len(splitted)

### Train Tokenizer and label encoder

In [208]:
tokenizer = Tokenizer(model = BPE(unk_token="<unk>"))
tokenizer.normalizer = Lowercase()
tokenizer.pre_tokenizer = Whitespace()

In [215]:
trainer = BpeTrainer(vocab_size=30_000, special_tokens=["<pad>", "<unk>"])
tokenizer.train_from_iterator(raw_train_dataset["text"], trainer=trainer)

In [216]:
main_tokenizer = PreTrainedTokenizerFast(tokenizer_object = tokenizer)

In [217]:
test_string = raw_train_dataset["text"][rd.randint(0, len(raw_train_dataset))]
result = main_tokenizer(test_string)["input_ids"]
result

[108, 409, 8, 509, 8, 416, 1583, 1123, 109, 108, 1704, 15]

In [218]:
main_tokenizer.decode(result)

"the movie ' inception ' won three awards at the oscars ."

In [323]:
all_labels = [item for label_list in clean_labels for item in label_list]
unique_labels = list(set(all_labels))
unique_labels

['I-FACILITY',
 'I-PHONE_NUMBER',
 'I-ADDRESS',
 'I-DATE',
 'B-DATE',
 'B-TIME',
 'I-NORP',
 'B-NUMBER',
 'B-LOCATION',
 'B-ORG',
 'I-LANGUAGE',
 'I-PRODUCT',
 'I-PERCENT',
 'I-ORG',
 'I-TIME',
 'B-NORP',
 'B-PHONE_NUMBER',
 'B-LANGUAGE',
 'B-PRODUCT',
 'I-NUMBER',
 'B-FACILITY',
 'I-WORK_OF_ART',
 'B-ADDRESS',
 'B-PERSON',
 'I-LOCATION',
 'B-WORK_OF_ART',
 'I-PERSON',
 'O',
 'B-PERCENT']

In [324]:
label_encoder = LabelEncoder()
label_encoder.fit(unique_labels)

Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)Holds the label for each class.","ndarray[<U14](29,)","['B-ADDRESS','B-DATE','B-FACILITY',...,'I-TIME','I-WORK_OF_ART','O']"


In [325]:
label_encoder.transform(['B-PERCENT']).item()

8

In [326]:
raw_train_dataset["text"][15]

'He drives a BMW 5 Series car.'

In [327]:
len(unique_labels)

29

In [362]:
len(set(label_encoder.transform(unique_labels).tolist()))

29

### Creation of PyTorch dataset

In [335]:
class NERDataset(Dataset):
    def __init__(self, text_data, label_data, text_encoder, label_encoder):
        self.text_data = text_data
        self.label_data = label_data
        self.tokenizer = text_encoder
        self.label_encoder = label_encoder
    
    def __len__(self):
        return len(self.text_data)
    
    def __getitem__(self, idx):
        splitted_text = self.text_data[idx]
        labels = self.label_data[idx]
        encoded_text = []
        encoded_labels = []
        for i, word in enumerate(splitted_text):
            encoded_word = self.tokenizer(word)['input_ids'] 
            if len(encoded_word) != 1:
                encoded_text.extend(encoded_word)
                for _ in range(len(encoded_word)):
                    encoded_labels.append(self.label_encoder.transform([labels[i]]).item())
            else:
                encoded_text.extend(encoded_word)
                encoded_labels.append(self.label_encoder.transform([labels[i]]).item())
        return torch.tensor(encoded_text), torch.tensor(encoded_labels)

In [336]:
dataset = NERDataset(splitted_text, clean_labels, main_tokenizer, label_encoder)
dataset

In [337]:
single_sample = dataset[181]
print(f"Length: {len(single_sample[0])}")
single_sample

Length: 16


(tensor([ 108,  461,  837, 7487, 2281,   13, 5253,   13, 5479, 9849,  115, 4822,
          127,   36, 1604, 1325]),
 tensor([28, 28, 28,  0, 14, 14,  4,  4,  4,  6, 28, 28, 28, 28, 13, 11]))

In [338]:
main_tokenizer.decode(single_sample[0])

'the address 789 zione lane , memphis , tn 38111 is listed as a historic site'

In [296]:
# check weather the lengths match
for x, y in tqdm(dataset, desc="Going through loader..."):
    assert len(x) == len(y), "Lengths do not match"

Going through loader...: 100%|██████████| 8716/8716 [00:24<00:00, 349.09it/s]


In [297]:
# Check max length of input
max_length = 0
for x, y in tqdm(dataset, desc="Going through loader..."):
    max_length = max(0, len(x))
max_length

Going through loader...: 100%|██████████| 8716/8716 [00:28<00:00, 309.88it/s]


21

In [312]:
single_sample = dataset[433]
len(single_sample[0])

9

In [313]:
label_encoder.inverse_transform([8])

array(['B-PERCENT'], dtype='<U14')

### Dataloader and Model training

In [339]:
dataloader = DataLoader(dataset = dataset, shuffle = True, batch_size = 1)
dataloader

In [355]:
dataloader_sample = next(iter(dataloader))
print(dataloader_sample[0].shape)
print(dataloader_sample[1].shape)
dataloader_sample

torch.Size([1, 16])
torch.Size([1, 16])


[tensor([[ 867,  773,  106, 3146, 2773, 1676, 2623,    6,  133, 2966, 9891,  123,
           108,  155, 1927,   15]]),
 tensor([[28, 28, 28, 13, 28, 28,  8,  8, 28, 28, 28, 28, 28, 11, 25, 25]])]

In [320]:
len(main_tokenizer.get_vocab())

11423

In [442]:
class NERmodel(nn.Module):
    def __init__(self, vocab_size = 11423, label_dict_size = 29, input_size= 300, hidden_size = 300):
        super().__init__()
        self.vocab_size = vocab_size
        self.label_dict_size = label_dict_size
        self.embedding = nn.Embedding(vocab_size, input_size)
        self.rnn = nn.LSTM(input_size = input_size, hidden_size=hidden_size, bidirectional=True)
        self.linear = nn.Linear(2*hidden_size, label_dict_size)
    
    def forward(self, x):
        y = self.embedding(x) # B x S x E
        y = y.transpose(0, 1)
        out, hc = self.rnn(y) # S x B x E
        y = out.transpose(0, 1)
        y = self.linear(y)
        return y.squeeze(dim = 0)

In [443]:
model = NERmodel()
model

NERmodel(
  (embedding): Embedding(11423, 300)
  (rnn): LSTM(300, 300, bidirectional=True)
  (linear): Linear(in_features=600, out_features=29, bias=True)
)

In [444]:
model(dataloader_sample[0]).shape

torch.Size([16, 29])

### Training

In [445]:
model = NERmodel().to(device = 'cuda')
model

NERmodel(
  (embedding): Embedding(11423, 300)
  (rnn): LSTM(300, 300, bidirectional=True)
  (linear): Linear(in_features=600, out_features=29, bias=True)
)

In [446]:
dataloader = DataLoader(dataset = dataset, shuffle = True, batch_size = 1)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.005, betas = (0.98, 0.99))

In [ ]:
epoch_losses = []
try:
    for epoch in range(1, 100 + 1):
        model.train()
        current_epoch_losses = []
        for inputs, targets in tqdm(dataloader, desc=f"Epoch {epoch} Train"):
            inputs, targets = inputs.to(device = 'cuda'), targets.to(device = 'cuda')
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets.flatten())
            loss.backward()
            optimizer.step()
            current_epoch_losses.append(loss.item())
        avg_loss = np.array(current_epoch_losses).mean()
        print(f"Current epoch #{epoch} | Loss: {avg_loss}")
        epoch_losses.append(avg_loss)
except KeyboardInterrupt:
    pass

Epoch 1 Train: 100%|██████████| 8716/8716 [01:40<00:00, 86.91it/s]


Current epoch #1 | Loss: 0.8457005785077897


Epoch 2 Train: 100%|██████████| 8716/8716 [01:39<00:00, 87.29it/s]


Current epoch #2 | Loss: 0.6794352204601264


Epoch 3 Train: 100%|██████████| 8716/8716 [01:39<00:00, 88.01it/s]


...

Epoch 29 Train: 100%|██████████| 8716/8716 [01:45<00:00, 82.83it/s]


Current epoch #29 | Loss: 0.4747839778027664


Epoch 30 Train: 100%|██████████| 8716/8716 [01:44<00:00, 83.79it/s]


Current epoch #30 | Loss: 0.4730547601285332


Epoch 31 Train:  39%|███▉      | 3406/8716 [00:43<01:04, 82.47it/s]

# Inference

In [395]:
# save previous model
torch.save(model.state_dict(), f="artifacts\\simpleLSTM.pth")